# Reinforcement Learning Control: Cart-Pole

This notebook demonstrates the reinforcement learning (RL) framework in **HILO-MPC**.
We use the classic **inverted pendulum on a cart** (cart-pole) as the benchmark system
and show how to:

1. Define the plant model with the `Model` class
2. Train a **Q-Learning** agent (tabular, discretized state space)
3. Train a **DQN** agent (deep Q-network with experience replay)
4. Compare the learned policies in closed-loop simulation

The RL interface mirrors the MPC controllers in HILO-MPC:

```python
agent = QLearningAgent(model)   # same pattern as NMPC(model)
agent.setup()                   # compile / initialise
u = agent.optimize(x0)          # get action for current state
```

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import casadi as ca

from hilo_mpc import Model, QLearningAgent, DQNAgent
from hilo_mpc import EpsilonGreedyPolicy

## 2. Cart-Pole Model

The nonlinear equations of motion for a cart-pole system:

$$\dot{x} = v$$
$$\dot{v} = \frac{1}{M + m - m\cos\theta} \left( m g \sin\theta - m l \sin\theta\, \omega^2 + F \right)$$
$$\dot{\theta} = \omega$$
$$\dot{\omega} = \frac{1}{l} \left( \ddot{x}\cos\theta + g\sin\theta \right)$$

where $(x, v)$ are cart position and velocity, $(\theta, \omega)$ are pole angle and angular velocity,
$F$ is the applied force, and $M, m, l$ are cart mass, pole mass, and pole length.

In [ ]:
# Physical constants
M = 5.0   # cart mass [kg]
m = 1.0   # pole mass [kg]
l = 1.0   # pole length [m]
g = 9.81  # gravity [m/s^2]
dt = 0.05 # sampling time [s]

model = Model(plot_backend=None)

# States: cart position, cart velocity, pole angle, pole angular velocity
x_sym = model.set_dynamical_states(['x', 'v', 'theta', 'omega'])
v, theta, omega = x_sym[1], x_sym[2], x_sym[3]

# Input: force applied to cart
F = model.set_inputs('F')

# Equations of motion
dx     = v
dv     = (1.0 / (M + m - m * ca.cos(theta))) * (
             m * g * ca.sin(theta) - m * l * ca.sin(theta) * omega**2 + F
         )
dtheta = omega
domega = (1.0 / l) * (dv * ca.cos(theta) + g * ca.sin(theta))

model.set_equations(ode=[dx, dv, dtheta, domega])
model.setup(dt=dt)

print(f"Model states : {model.dynamical_state_names}")
print(f"Model inputs : {model.input_names}")
print(f"Sampling time: {dt} s")

## 3. Reward Function

We penalise the pole angle and cart displacement from the upright equilibrium
$(x, v, \theta, \omega) = (0, 0, 0, 0)$:

In [ ]:
def reward_fn(x, u, x_next):
    """Reward = negative weighted sum of squared deviations from upright."""
    x_pos, v_cart, theta, omega = x_next
    return -(10.0 * theta**2 + 1.0 * omega**2 + 0.1 * x_pos**2 + 0.1 * float(u)**2)


def is_done(x):
    """Episode ends if pole falls past ±30° or cart leaves ±3 m."""
    return abs(x[2]) > np.deg2rad(30) or abs(x[0]) > 3.0

## 4. Q-Learning Agent

Tabular Q-learning works by discretising the continuous state space.
We use 10 bins per dimension covering the expected operating range.

In [ ]:
# Discrete action set: push left, do nothing, push right
actions = [-10.0, 0.0, 10.0]  # [N]

ql_agent = QLearningAgent(model)

# Discrete action set
ql_agent.set_action_space(actions)

# State-space discretisation: (low, high) for each state, number of bins
ql_agent.set_state_space(
    bounds=[(-3.0, 3.0),           # cart position [m]
            (-5.0, 5.0),           # cart velocity [m/s]
            (-0.6, 0.6),           # pole angle [rad]  (~±34°)
            (-2.0, 2.0)],          # angular velocity [rad/s]
    n_bins=10
)

ql_agent.set_reward_function(reward_fn)

# Hyper-parameters
ql_agent.learning_rate   = 0.2
ql_agent.discount_factor = 0.99
ql_agent.policy = EpsilonGreedyPolicy(epsilon=1.0, epsilon_min=0.02,
                                       epsilon_decay=0.998)

ql_agent.setup()
print(f"Q-table shape: {ql_agent.q_table.shape}")
print(f"Agent type   : {ql_agent.type}")

In [ ]:
# Training
x0 = [0.0, 0.0, 0.1, 0.0]   # small initial pole tilt
N_EPISODES_QL = 300
N_STEPS       = 200

ql_rewards = ql_agent.train(
    x0=x0, n_steps=N_STEPS, n_episodes=N_EPISODES_QL, done_fn=is_done
)

print(f"First 5 episode rewards : {[f'{r:.1f}' for r in ql_rewards[:5]]}")
print(f"Last  5 episode rewards : {[f'{r:.1f}' for r in ql_rewards[-5:]]}")

In [ ]:
# Smooth the learning curve with a rolling mean
window = 20
ql_smooth = np.convolve(ql_rewards, np.ones(window) / window, mode='valid')

plt.figure(figsize=(8, 3))
plt.plot(ql_rewards, alpha=0.3, label='Episode reward')
plt.plot(np.arange(window - 1, len(ql_rewards)), ql_smooth, label=f'{window}-ep mean')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('Q-Learning: training curve')
plt.legend()
plt.tight_layout()
plt.show()

## 5. DQN Agent

The Deep Q-Network agent uses a small feed-forward neural network to approximate
the Q-function.  Unlike tabular Q-learning it does **not** need a discretised
state space and generalises better to unseen states.

In [ ]:
dqn_agent = DQNAgent(
    model,
    hidden_layers=(64, 64),   # two hidden layers with 64 units each
    batch_size=32,
    buffer_size=5000,
    target_update_freq=50
)

dqn_agent.set_action_space(actions)
dqn_agent.set_reward_function(reward_fn)

dqn_agent.learning_rate   = 1e-3
dqn_agent.discount_factor = 0.99
dqn_agent.policy = EpsilonGreedyPolicy(epsilon=1.0, epsilon_min=0.02,
                                        epsilon_decay=0.997)

dqn_agent.setup()
print(f"Agent type          : {dqn_agent.type}")
print(f"Replay buffer size  : {dqn_agent._buffer_size}")

In [ ]:
N_EPISODES_DQN = 200

dqn_rewards = dqn_agent.train(
    x0=x0, n_steps=N_STEPS, n_episodes=N_EPISODES_DQN, done_fn=is_done
)

print(f"First 5 episode rewards : {[f'{r:.1f}' for r in dqn_rewards[:5]]}")
print(f"Last  5 episode rewards : {[f'{r:.1f}' for r in dqn_rewards[-5:]]}")

In [ ]:
dqn_smooth = np.convolve(dqn_rewards, np.ones(window) / window, mode='valid')

plt.figure(figsize=(8, 3))
plt.plot(dqn_rewards, alpha=0.3, label='Episode reward')
plt.plot(np.arange(window - 1, len(dqn_rewards)), dqn_smooth, label=f'{window}-ep mean')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.title('DQN: training curve')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Closed-Loop Evaluation

After training we run both agents for a fixed number of steps using
`agent.optimize(x0)` — the same call pattern as `nmpc.optimize(x0)`.

In [ ]:
def rollout(agent, x0, n_steps=150):
    """Run one closed-loop episode and record trajectories."""
    model.set_initial_conditions(x0=list(x0))
    x = np.array(x0, dtype=float)
    xs, us = [x.copy()], []
    for _ in range(n_steps):
        u = agent.optimize(x)
        model.simulate(u=u.flatten().tolist())
        x = np.array(model.solution['x:f'], dtype=float).flatten()
        xs.append(x.copy())
        us.append(float(u))
    return np.array(xs), np.array(us)


x0_eval = [0.0, 0.0, 0.15, 0.0]   # 8.6° initial tilt

xs_ql,  us_ql  = rollout(ql_agent,  x0_eval)
xs_dqn, us_dqn = rollout(dqn_agent, x0_eval)

t = np.arange(len(xs_ql)) * dt

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

# Pole angle
axes[0].plot(t, np.rad2deg(xs_ql[:, 2]),  label='Q-Learning')
axes[0].plot(t, np.rad2deg(xs_dqn[:, 2]), label='DQN', linestyle='--')
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].set_ylabel('Pole angle [°]')
axes[0].legend()

# Cart position
axes[1].plot(t, xs_ql[:, 0],  label='Q-Learning')
axes[1].plot(t, xs_dqn[:, 0], label='DQN', linestyle='--')
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].set_ylabel('Cart position [m]')

# Control force
axes[2].step(t[:-1], us_ql,  where='post', label='Q-Learning')
axes[2].step(t[:-1], us_dqn, where='post', label='DQN', linestyle='--')
axes[2].axhline(0, color='k', linewidth=0.5)
axes[2].set_ylabel('Force [N]')
axes[2].set_xlabel('Time [s]')

fig.suptitle('Closed-loop performance: Q-Learning vs DQN', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Using a Custom Policy

All policy classes implement the same `Policy` interface and can be swapped in
at any time.  Here we show the `SoftmaxPolicy` for temperature-based exploration:

In [ ]:
from hilo_mpc import SoftmaxPolicy

ql_softmax = QLearningAgent(model)
ql_softmax.set_action_space(actions)
ql_softmax.set_state_space(
    bounds=[(-3.0, 3.0), (-5.0, 5.0), (-0.6, 0.6), (-2.0, 2.0)],
    n_bins=10
)
ql_softmax.set_reward_function(reward_fn)
ql_softmax.learning_rate   = 0.2
ql_softmax.discount_factor = 0.99

# Use Softmax policy instead of epsilon-greedy
ql_softmax.policy = SoftmaxPolicy(temperature=2.0)

ql_softmax.setup()
rewards_sm = ql_softmax.train(x0=x0, n_steps=N_STEPS, n_episodes=100,
                               done_fn=is_done)

plt.figure(figsize=(7, 3))
plt.plot(rewards_sm, alpha=0.4, label='Episode reward')
sm_smooth = np.convolve(rewards_sm, np.ones(10) / 10, mode='valid')
plt.plot(np.arange(9, len(rewards_sm)), sm_smooth, label='10-ep mean')
plt.title('Q-Learning with SoftmaxPolicy')
plt.xlabel('Episode')
plt.ylabel('Total reward')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Saving and Loading a Trained Policy

The Q-Learning agent can persist its Q-table:

In [ ]:
ql_agent.save('/tmp/cartpole_qtable.npy')
print('Q-table saved to /tmp/cartpole_qtable.npy')

# Reload into a fresh agent
ql_loaded = QLearningAgent(model)
ql_loaded.set_action_space(actions)
ql_loaded.set_state_space(
    bounds=[(-3.0, 3.0), (-5.0, 5.0), (-0.6, 0.6), (-2.0, 2.0)],
    n_bins=10
)
ql_loaded.set_reward_function(reward_fn)
ql_loaded.setup()
ql_loaded.load('/tmp/cartpole_qtable.npy')

# Verify identical Q-tables
import numpy as np
assert np.allclose(ql_agent.q_table, ql_loaded.q_table)
print('Loaded Q-table matches the original.')

## Summary

| Feature | Detail |
|---|---|
| **Interface** | `__init__(model)` → `setup()` → `optimize(x0)` — same as MPC |
| **QLearningAgent** | Tabular, works with any `Model`; needs `set_state_space()` |
| **DQNAgent** | Neural-network Q-function; no extra ML library required |
| **Policies** | Swap `EpsilonGreedyPolicy`, `GreedyPolicy`, `SoftmaxPolicy` freely |
| **Extensibility** | Sub-class `RLBase` and override `setup()`, `optimize()`, `update()` |

To add a new RL algorithm (e.g., PPO, SAC) simply sub-class `RLBase` and implement
the three abstract methods.